In [0]:
CREATE TABLE IF NOT EXISTS isp.employee.agent (
		agent_id STRING NOT NULL,
		first_name STRING,
		last_name STRING,
		employment_date DATE,
		experience_level STRING,
		monthly_salary_eur INTEGER,
		CONSTRAINT agent_pk PRIMARY KEY (agent_id)
	);

-- Edit constraints of table
ALTER TABLE
	isp.employee.agent
DROP CONSTRAINT IF EXISTS
	agent_ed;

ALTER TABLE
	isp.employee.agent
DROP CONSTRAINT IF EXISTS
	agent_ms;

ALTER TABLE
	isp.employee.agent
ADD
	CONSTRAINT agent_ed CHECK (employment_date <= current_date());

ALTER TABLE
	isp.employee.agent
ADD
	CONSTRAINT agent_ms CHECK (monthly_salary_eur >= 0);

-- Add table for deleted values
MERGE INTO
	isp.employee.agent a
USING (
	SELECT
		*
	FROM
		isp.employee.agent_delete
) d
ON
	a.agent_id = d.agent_id
WHEN MATCHED THEN DELETE;

-- Fill table with values
MERGE INTO
	isp.employee.agent s
USING (
	SELECT
		a.agent_id,
		substring(a.agent_name, 1, POSITION(' ' IN a.agent_name) - 1) AS first_name,
		substring(a.agent_name, POSITION(' ' IN a.agent_name) + 1) AS last_name,
		to_date(a.employment_date) AS employment_date,
		a.experience_level,
		CAST(a.monthly_salary_eur AS INT) AS monthly_salary_eur
	FROM
		isp.employee.agent_bronze a
			LEFT JOIN isp.employee.agent_delete d
				ON a.agent_id = d.agent_id
	WHERE
		d.agent_id IS NULL
	QUALIFY
		row_number() OVER (PARTITION BY a.agent_id ORDER BY a.ingestion_time DESC) = 1
) b
ON
	s.agent_id = b.agent_id
WHEN MATCHED AND
	sha1(
		concat_ws(
			'|',
			b.first_name,
			b.last_name,
			b.employment_date,
			b.experience_level,
			b.monthly_salary_eur
		)
	)
		!= sha1(
			concat_ws(
				'|',
				s.first_name,
				s.last_name,
				s.employment_date,
				s.experience_level,
				s.monthly_salary_eur
			)
		)
	THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;